In [2]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/ammara-asif/fyp.git"   # <-- EDIT ONCE
BRANCH = "main"

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, CODE], check=True)
print(subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
for _m in ["fyp_infra", "stage1_runner"]:
    sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop: a CPU run silently takes ~3x longer and its timings are useless for budgeting.
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
c553fc5 stage1: lr grid



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ammara-asif125 (kashaf-saleem09-institute-of-business-administration) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'root': '/content/drive/MyDrive/fyp_nested_cl', 'data': '/content/drive/MyDrive/fyp_nested_cl/data', 'results': '/content/drive/MyDrive/fyp_nested_cl/results', 'logs': '/content/drive/MyDrive/fyp_nested_cl/logs', 'checkpoints': '/content/drive/MyDrive/fyp_nested_cl/checkpoints'}
GPU: Tesla T4


In [3]:
EPOCHS = 10
LRS = [0.01, 0.03, 0.1]
ARMS = {
    "flat":      dict(lr_schedule="flat"),
    "flat_llrd": dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="slow_deep"),
}

def grid_cfg(arm, lr):
    # The flat/0.03 point already exists as the epoch-sweep run; keeping its stage name
    # gives the identical config -> identical run_id -> run_sweep reuses that result.
    stage = "stage1_epoch_sweep" if (arm == "flat" and lr == 0.03) else "stage1_lr_grid"
    return sr.make_cfg(stage=stage, strategy="naive", epochs=EPOCHS, seed=0, lr=lr,
                       eval_split="val", **ARMS[arm])

CONFIGS = [grid_cfg(arm, lr) for arm in ARMS for lr in LRS]
SHOW = ["lr_schedule", "lr", "epochs", "eval_split"]
for c in CONFIGS: print(run_id(c), {k: c[k] for k in SHOW})

8965e350a6ae {'lr_schedule': 'flat', 'lr': 0.01, 'epochs': 10, 'eval_split': 'val'}
714a54d5fe81 {'lr_schedule': 'flat', 'lr': 0.03, 'epochs': 10, 'eval_split': 'val'}
c5e3da3ce9a2 {'lr_schedule': 'flat', 'lr': 0.1, 'epochs': 10, 'eval_split': 'val'}
f6325928117a {'lr_schedule': 'llrd', 'lr': 0.01, 'epochs': 10, 'eval_split': 'val'}
e4a01a3faae5 {'lr_schedule': 'llrd', 'lr': 0.03, 'epochs': 10, 'eval_split': 'val'}
f13c2d5c3a20 {'lr_schedule': 'llrd', 'lr': 0.1, 'epochs': 10, 'eval_split': 'val'}


In [4]:
out = run_sweep(CONFIGS, sr.run_experiment, paths)

[1/6] 8965e350a6ae running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")


  exp 1/20: seen-avg 66.00%  current 66.00%  train-loss 1.745
  exp 2/20: seen-avg 40.20%  current 80.40%  train-loss 1.527
  exp 3/20: seen-avg 18.27%  current 54.80%  train-loss 2.029
  exp 4/20: seen-avg 19.50%  current 78.00%  train-loss 1.701
  exp 5/20: seen-avg 11.52%  current 57.60%  train-loss 2.123
  exp 6/20: seen-avg 10.67%  current 64.00%  train-loss 2.119
  exp 7/20: seen-avg 10.34%  current 72.40%  train-loss 1.782
  exp 8/20: seen-avg 8.70%  current 69.60%  train-loss 1.778
  exp 9/20: seen-avg 6.84%  current 61.60%  train-loss 2.036
  exp 10/20: seen-avg 6.88%  current 68.80%  train-loss 1.859
  exp 11/20: seen-avg 5.89%  current 64.80%  train-loss 1.987
  exp 12/20: seen-avg 5.10%  current 61.20%  train-loss 2.080
  exp 13/20: seen-avg 4.34%  current 56.40%  train-loss 2.238
  exp 14/20: seen-avg 4.80%  current 67.20%  train-loss 2.061
  exp 15/20: seen-avg 4.27%  current 64.00%  train-loss 2.235
  exp 16/20: seen-avg 4.38%  current 70.00%  train-loss 2.010
  exp 17/2

acc_current_exp,▄█▁▇▂▄▆▅▃▅▄▃▁▄▄▅▇▃▅▃
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


[2/6] 714a54d5fe81 already finished, skipping
[3/6] c5e3da3ce9a2 running


...

acc_current_exp,▄█▁▆▂▃▅▆▂▇▆▄▃▃▆▅▇▅▆▆
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


[6/6] f13c2d5c3a20 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")


  exp 1/20: seen-avg 73.60%  current 73.60%  train-loss 1.096
  exp 2/20: seen-avg 41.20%  current 82.40%  train-loss 0.794
  exp 3/20: seen-avg 23.07%  current 69.20%  train-loss 1.137
  exp 4/20: seen-avg 21.10%  current 84.40%  train-loss 0.840
  exp 5/20: seen-avg 12.72%  current 63.60%  train-loss 1.195
  exp 6/20: seen-avg 10.20%  current 61.20%  train-loss 1.064
  exp 7/20: seen-avg 11.31%  current 79.20%  train-loss 0.937
  exp 8/20: seen-avg 10.10%  current 80.80%  train-loss 0.768
  exp 9/20: seen-avg 8.36%  current 75.20%  train-loss 0.945
  exp 10/20: seen-avg 8.32%  current 83.20%  train-loss 0.885
  exp 11/20: seen-avg 6.98%  current 76.80%  train-loss 0.952
  exp 12/20: seen-avg 5.97%  current 71.60%  train-loss 0.955
  exp 13/20: seen-avg 5.66%  current 73.60%  train-loss 1.138
  exp 14/20: seen-avg 5.91%  current 82.80%  train-loss 0.880
  exp 15/20: seen-avg 5.39%  current 80.80%  train-loss 0.921
  exp 16/20: seen-avg 5.30%  current 84.80%  train-loss 0.778
  exp 17/

acc_current_exp,▅▇▃█▂▁▆▇▅▇▆▄▅▇▇██▇▄▆
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


In [5]:
import numpy as np, pandas as pd
rows = []
for cfg in CONFIGS:
    r = load_result(cfg, paths)
    if r is None:
        rows.append({**{k: cfg[k] for k in SHOW}, "status": "not finished"}); continue
    m = r["metrics"]
    rows.append({**{k: cfg[k] for k in SHOW}, "run_id": r["run_id"],
                 "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
                 "avg_incremental_acc": m["avg_incremental_acc"],
                 # mean of the diagonal: how well each task was learned when it was current.
                 # Report next to forgetting: forgetting = peak - final, so it grows with this.
                 "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
                 "steps": m["optimizer_steps"], "n_params": m["n_params"], "minutes": r["wall_clock_s"] / 60})
df = pd.DataFrame(rows); df

,lr_schedule,lr,epochs,eval_split,run_id,final_avg_acc,avg_forgetting,avg_incremental_acc,acc_just_learned,steps,n_params,minutes
0,flat,0.01,10,val,8965e350a6ae,3.16,66.568421,12.122676,66.40,14200,1109240,8.982242
1,flat,0.03,10,val,714a54d5fe81,3.90,75.978947,13.468379,76.08,14200,1109240,117.254350
2,flat,0.10,10,val,c5e3da3ce9a2,4.44,81.368421,14.474445,81.74,14200,1109240,8.707282
3,llrd,0.01,10,val,f6325928117a,2.84,56.084211,9.944344,56.12,14200,1109240,8.466168
4,llrd,0.03,10,val,e4a01a3faae5,3.60,68.547368,12.299203,68.72,14200,1109240,8.473640
5,llrd,0.10,10,val,f13c2d5c3a20,3.96,77.031579,13.628372,77.14,14200,1109240,8.565495


In [6]:
best = df.loc[df.groupby("lr_schedule")["avg_incremental_acc"].idxmax(), ["lr_schedule", "lr", "avg_incremental_acc"]]
print(best)
for _, r in best.iterrows():
    if r["lr"] in (min(LRS), max(LRS)):
        print(f"WARNING: best lr for {r['lr_schedule']} is at the grid edge ({r['lr']}); extend the grid before choosing.")

  lr_schedule   lr  avg_incremental_acc
2        flat  0.1            14.474445
5        llrd  0.1            13.628372
